In [2]:
# ==========================================
# CELLULE 1 : IMPORTATIONS & CONFIGURATION
# ==========================================
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

# Configuration de l'appareil de calcul (CPU)
device = "cpu"
print(f"Calculs configurés sur : {device}")

Calculs configurés sur : cpu


In [3]:
# ==========================================
# CELLULE 2 : CHARGEMENT DES DONNÉES (MNIST)
# ==========================================
# Téléchargement des données d'entraînement et de test
training_data = datasets.FashionMNIST(
    root="data", train=True, download=True, transform=ToTensor()
)
test_data = datasets.FashionMNIST(
    root="data", train=False, download=True, transform=ToTensor()
)

# Création des DataLaoders (Gestion Automatique des Mini-Batchs)
# C'est ici qu'on définit que chaque itération traitera 64 images d'un coup !
batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in train_dataloader:
    print(f"Forme de X (Batch, Canaux, H, W) : {X.shape}")
    print(f"Forme de y (Labels du Batch) : {y.shape} (ex: {y[:5].tolist()}...)")
    break

100.0%
100.0%
100.0%
100.0%

Forme de X (Batch, Canaux, H, W) : torch.Size([64, 1, 28, 28])
Forme de y (Labels du Batch) : torch.Size([64]) (ex: [9, 0, 0, 3, 0]...)


In [4]:
# ==========================================
# CELLULE 3 : DÉFINITION DE L'ARCHITECTURE
# ==========================================
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten() # Aplatit (64, 1, 28, 28) -> (64, 784)
        
        # nn.Sequential automatise le flux du forward
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512), # Couche d'entrée dense
            nn.ReLU(),             # Règle le problème du Vanishing Gradient
            nn.Linear(512, 512),   # Couche cachée intermédiaire
            nn.ReLU(),
            nn.Linear(512, 10),    # Couche de sortie : 10 LOGITS bruts
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

# Instanciation de notre classe concrète
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
# ==========================================
# CELLULE 4 : LOSS FUNCTION & OPTIMISEUR
# ==========================================
# 1. LA CROSS ENTROPY LOSS (2-en-1)
# Rappel : Elle applique le Softmax en interne, puis calcule la perte log.
# On lui donne les LOGITS bruts directement en sortie du modèle.
loss_fn = nn.CrossEntropyLoss()

# 2. L'OPTIMISEUR (Mini-Batch SGD avec inertie)
# Il ne connaît pas la batch_size, il applique juste la formule sur le gradient moyen.
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

In [6]:
# ==========================================
# CELLULE 5 : LA BOUCLE D'ENTRAÎNEMENT (4 TEMPS)
# ==========================================
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train() # Mode entraînement active l'Autograd
    
    # dataloader nous donne les paquets (Mini-Batchs) un par un
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # ---- LE RYTHME STRICT À 4 TEMPS DE PYTORCH ----
        
        # 1. On efface les gradients accumulés au batch précédent
        optimizer.zero_grad()
        
        # 2. Forward pass : Calcul des prédictions (logits) et de la perte
        pred = model(X)
        loss = loss_fn(pred, y)
        
        # 3. Backward pass : Rétropropagation unique de l'erreur dans le graphe
        loss.backward()
        
        # 4. Step : L'optimiseur met à jour les poids (model.parameters())
        optimizer.step()

        # Affichage du statut toutes les 100 itérations
        if batch % 100 == 0:
            loss_val, current = loss.item(), batch * len(X)
            print(f"Perte (Loss): {loss_val:>7f}  [{current:>5d}/{size:>5d}]")

# Fonction pour évaluer le modèle sur les données de Test (qu'il n'a jamais vues)
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval() # Mode évaluation (désactive certains calculs d'Autograd)
    test_loss, correct = 0, 0

    with torch.no_grad(): # Bloque la création du graphe dynamique (gain de RAM)
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"\nRésultats Test: \n Précision (Accuracy): {(100*correct):>0.1f}%, Perte Moyenne: {test_loss:>8f} \n")

In [7]:
# ==========================================
# CELLULE 6 : LE LANCEMENT DU CALCUL (ÉPOQUES)
# ==========================================
# Une époque = tout le dataset est passé une fois à travers le réseau
epochs = 3
for t in range(epochs):
    print(f"-------------------------------\nÉPOQUE {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Entraînement Terminé !")

-------------------------------
ÉPOQUE 1
-------------------------------
Perte (Loss): 2.306549  [    0/60000]
Perte (Loss): 0.946118  [ 6400/60000]
Perte (Loss): 0.547523  [12800/60000]
Perte (Loss): 0.673872  [19200/60000]
Perte (Loss): 0.559487  [25600/60000]
Perte (Loss): 0.529327  [32000/60000]
Perte (Loss): 0.490494  [38400/60000]
Perte (Loss): 0.623935  [44800/60000]
Perte (Loss): 0.562338  [51200/60000]
Perte (Loss): 0.509201  [57600/60000]

Résultats Test: 
 Précision (Accuracy): 82.5%, Perte Moyenne: 0.487751 

-------------------------------
ÉPOQUE 2
-------------------------------
Perte (Loss): 0.354760  [    0/60000]
Perte (Loss): 0.435405  [ 6400/60000]
Perte (Loss): 0.319694  [12800/60000]
Perte (Loss): 0.487101  [19200/60000]
Perte (Loss): 0.422759  [25600/60000]
Perte (Loss): 0.437285  [32000/60000]
Perte (Loss): 0.388695  [38400/60000]
Perte (Loss): 0.527634  [44800/60000]
Perte (Loss): 0.484648  [51200/60000]
Perte (Loss): 0.505187  [57600/60000]

Résultats Test: 
 P